# Controlled Stage-2 baseline: Adam vs Canonical SOAP

This Kaggle notebook runs the first controlled comparison for the 2-D Poisson PINN. It is self-contained and does not require a local Python environment.

It creates one replayable 2000-step batch schedule, then runs the 100-step prefix with:

- Adam
- Canonical SOAP (the official `nikhilvyas/SOAP` implementation)

Both runs use seed 0, K=1, the same Xavier-initialized 4-hidden-layer x 128 Tanh model, and the same frozen batches. JSON outputs and the schedule are written to `/kaggle/working/results/`.

## Kaggle setup

1. Open this notebook in Kaggle and enable **Settings > Accelerator > GPU**.
2. Enable Kaggle Internet access so the notebook can clone the official SOAP repository.
3. Run every cell from top to bottom.
4. Download `stage2_adam_seed0_K1_100.json`, `stage2_soap_seed0_K1_100.json`, and `poisson_seed0_K1_schedule.pt` from the final cell.

The notebook does not implement paper-SOAP-style reconstruction or MoQ. Canonical SOAP here means the unmodified official implementation.

In [ ]:
from __future__ import annotations

import json
import math
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn

# Kaggle writes downloadable artifacts to this directory.
output_dir = Path('/kaggle/working/results')
output_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch: {torch.__version__}')

## Frozen experiment configuration

This cell is the only configuration cell for tonight's smoke tests. The schedule is always generated for 2000 steps so that later 500- and 2000-step runs use the same prefix.

In [ ]:
# ===== Frozen Stage-2 setup =====
steps = 100
schedule_steps = 2000
seed = 0
K = 1
n_pde = 1000
n_boundary_each = 100
lr = 1e-3
eval_every = 100
schedule_path = output_dir / f'poisson_seed{seed}_K{K}_schedule.pt'

print({
    'steps': steps,
    'schedule_steps': schedule_steps,
    'seed': seed,
    'K': K,
    'n_pde': n_pde,
    'n_boundary_each': n_boundary_each,
    'lr': lr,
    'schedule': str(schedule_path),
})

In [ ]:
# ===== Install/copy the official SOAP implementation inside Kaggle =====
soap_repo = Path('/kaggle/working/SOAP')
soap_source = Path('/kaggle/working/soap_official.py')
if not soap_source.exists():
    if not soap_repo.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/nikhilvyas/SOAP.git', str(soap_repo)],
            check=True,
        )
    shutil.copyfile(soap_repo / 'soap.py', soap_source)

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')
from soap_official import SOAP

soap_commit = subprocess.check_output(
    ['git', '-C', str(soap_repo), 'rev-parse', 'HEAD'],
    text=True,
).strip() if soap_repo.exists() else 'unknown'
print(f'Official SOAP import OK; commit={soap_commit}')


class PINN(nn.Module):
    """Four-hidden-layer Tanh MLP with width 128."""

    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)


def set_seed(seed_value: int) -> None:
    torch.manual_seed(seed_value)
    np.random.seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)


def sample_boundary(n_each: int) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    boundary = torch.empty(4 * n_each, 2, device=device)
    boundary[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    boundary[n_each:2 * n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2 * n_each]], dim=1)
    boundary[2 * n_each:3 * n_each] = torch.stack([t[2 * n_each:3 * n_each], torch.zeros(n_each, device=device)], dim=1)
    boundary[3 * n_each:] = torch.stack([t[3 * n_each:], torch.ones(n_each, device=device)], dim=1)
    return boundary


def make_batch_schedule(schedule_length: int) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    boundary: Optional[torch.Tensor] = None
    for step in range(1, schedule_length + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = torch.rand(n_pde, 2, device=device, requires_grad=True)
            boundary = sample_boundary(n_boundary_each)
        assert pde is not None and boundary is not None
        schedule.append({'pde': pde.detach().clone(), 'bc': boundary.detach().clone()})
    return schedule


if schedule_path.exists():
    schedule = torch.load(schedule_path, map_location=device)
    if len(schedule) < schedule_steps:
        raise ValueError(f'Schedule has {len(schedule)} entries; expected {schedule_steps}')
    print(f'Loaded frozen schedule: {schedule_path}')
else:
    set_seed(seed)
    schedule = make_batch_schedule(schedule_steps)
    torch.save(schedule, schedule_path)
    print(f'Saved frozen schedule: {schedule_path}')

In [ ]:
def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    prediction = model(xy)
    first_derivatives = torch.autograd.grad(
        prediction, xy, grad_outputs=torch.ones_like(prediction), create_graph=True, retain_graph=True
    )[0]
    u_x = first_derivatives[:, 0:1]
    u_y = first_derivatives[:, 1:2]
    u_xx = torch.autograd.grad(
        u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True
    )[0][:, 0:1]
    u_yy = torch.autograd.grad(
        u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True
    )[0][:, 1:2]
    forcing = 2.0 * math.pi ** 2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing


def relative_l2(model: nn.Module) -> float:
    grid = torch.linspace(0.0, 1.0, 81, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing='ij')
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        prediction = model(xy).reshape(-1)
    reference = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(prediction - reference) / (torch.linalg.vector_norm(reference) + 1e-12))


def flat_grad(model: nn.Module) -> torch.Tensor:
    gradients = [parameter.grad.detach().reshape(-1) for parameter in model.parameters() if parameter.grad is not None]
    if not gradients:
        raise RuntimeError('No gradients available')
    return torch.cat(gradients)


def make_optimizer(name: str, model: nn.Module) -> torch.optim.Optimizer:
    if name == 'adam':
        return torch.optim.Adam(model.parameters(), lr=lr)
    if name == 'soap':
        return SOAP(
            model.parameters(),
            lr=lr,
            betas=(0.95, 0.95),
            weight_decay=0.0,
            precondition_frequency=10,
        )
    raise ValueError(name)


def run_experiment(optimizer_name: str, output_name: str) -> Dict[str, object]:
    # Reset RNG before each model so Adam and SOAP start from identical theta_0.
    set_seed(seed)
    model = PINN().to(device)
    optimizer_instance = make_optimizer(optimizer_name, model)
    previous_gradient: Optional[torch.Tensor] = None
    previous_gradient_norm: Optional[float] = None
    history: List[Dict[str, float]] = []
    start_time = time.perf_counter()

    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]['pde'].detach().clone().requires_grad_(True)
        current_boundary = schedule[step - 1]['bc']
        optimizer_instance.zero_grad(set_to_none=True)
        residual = poisson_residual(model, current_pde)
        loss_pde = torch.mean(residual.square())
        loss_bc = torch.mean(model(current_boundary).square())
        loss = loss_pde + loss_bc
        loss.backward()

        gradient = flat_grad(model)
        gradient_norm = float(torch.linalg.vector_norm(gradient))
        gradient_cosine = float('nan')
        gradient_difference_norm = float('nan')
        gradient_change_ratio = float('nan')
        if previous_gradient is not None and previous_gradient_norm is not None:
            gradient_difference = gradient - previous_gradient
            gradient_difference_norm = float(torch.linalg.vector_norm(gradient_difference))
            gradient_change_ratio = gradient_difference_norm / (gradient_norm + 1e-12)
            gradient_cosine = float(
                torch.dot(gradient, previous_gradient)
                / (gradient_norm * previous_gradient_norm + 1e-12)
            )

        before_update = torch.cat([parameter.detach().reshape(-1) for parameter in model.parameters()])
        optimizer_instance.step()
        after_update = torch.cat([parameter.detach().reshape(-1) for parameter in model.parameters()])
        update_norm = float(torch.linalg.vector_norm(after_update - before_update))
        previous_gradient = gradient.clone()
        previous_gradient_norm = gradient_norm

        if step == 1 or step % eval_every == 0 or step == steps:
            row = {
                'step': step,
                'loss': float(loss.detach()),
                'loss_pde': float(loss_pde.detach()),
                'loss_bc': float(loss_bc.detach()),
                'rel_l2': relative_l2(model),
                'grad_norm': gradient_norm,
                'grad_cos_prev': gradient_cosine,
                'grad_diff_norm': gradient_difference_norm,
                'grad_change_ratio': gradient_change_ratio,
                'update_norm': update_norm,
                'elapsed_sec': time.perf_counter() - start_time,
            }
            history.append(row)
            print(
                f'[{optimizer_name}] step={step:3d} loss={row["loss"]:.4e} '
                f'relL2={row["rel_l2"]:.4e} update={update_norm:.4e}'
            )

        if not torch.isfinite(after_update).all():
            raise FloatingPointError(f'{optimizer_name} parameters became non-finite at step {step}')

    result: Dict[str, object] = {
        'stage': 2,
        'label': 'Canonical SOAP' if optimizer_name == 'soap' else 'Adam',
        'optimizer': optimizer_name,
        'seed': seed,
        'K': K,
        'steps': steps,
        'schedule_steps': schedule_steps,
        'n_pde': n_pde,
        'n_boundary_each': n_boundary_each,
        'lr': lr,
        'device': str(device),
        'torch_version': torch.__version__,
        'soap_commit': soap_commit,
        'history': history,
    }
    result_path = output_dir / output_name
    result_path.write_text(json.dumps(result, indent=2))
    print(f'Saved: {result_path}')
    return result


adam_result = run_experiment('adam', 'stage2_adam_seed0_K1_100.json')
soap_result = run_experiment('soap', 'stage2_soap_seed0_K1_100.json')

In [ ]:
from IPython.display import FileLink, display

for filename in ['stage2_adam_seed0_K1_100.json', 'stage2_soap_seed0_K1_100.json', schedule_path.name]:
    artifact = output_dir / filename
    print(f'{filename}: {artifact.stat().st_size} bytes')
    display(FileLink(str(artifact)))

print('\nFinal Adam row:', adam_result['history'][-1])
print('Final Canonical SOAP row:', soap_result['history'][-1])